# O+O vacuum closure (R4), the p+p reference for O+O (R3) and a first look at the centrality bias (R4b)

Steps R3 and R4 of the plan forward on the tuning page (section *Plan forward*). Produced by
`closure_oo.py` (run + analyse); model results, not vetted physics results.

## Why this test
In the model, $R_{AA}$ divides the O+O spectrum by $N_\mathrm{coll}$ times the p+p spectrum of the **same
chain**. Everything that is not the medium should then cancel. The vacuum closure checks that it does:
O+O with the medium switched off (`--vacuum`: no hydro, no LBT, Matter in vacuum, iColored) against
p+p from the same XML. Any deviation from 1 is a non-medium effect that the O+O $R_{AA}$ would otherwise
silently contain — the nucleons' other collisions, the momentum the hard scattering takes from them
(the hard-momentum subtraction), their remnants and the colour flow into iColored. At 200 GeV a 20 GeV
jet takes a large share of a nucleon's energy, so energy conservation could matter here far more than at
the LHC.

## What the generator does, and so what this test can and cannot see
- Every event has **exactly one hard scattering**, generated by Pythia for a free p+p pair (`Beams:idA/idB =
  2212` in `PythiaIsrGun`), placed at one of the event's binary collisions. Its momentum is subtracted
  from the two nucleons; what 3dMCGlauber leaves of them (the remnants) goes to iColored with the shower.
- So **$N_\mathrm{coll}$ scaling is built in**, and neither isospin nor energy conservation can veto or
  reshape the hard scattering itself. The comparison is per hard scattering: the cross-section-weighted
  spectra $\sum_k \sigma_k/N_k\,w_k$ of O+O and p+p, in the same $\hat p_T$ windows.
- What it **can** see: changes of the hadrons and jets of the hard event from the O+O environment
  (remnants, colour flow, iColored), events that fail, and how often 3dMCGlauber's bookkeeping cannot
  accommodate the hard momentum (its "Can not find x > x_hard" message).
- What it **cannot** see, by construction: the **centrality bias** (with impact-parameter classes and no
  bulk there is no multiplicity to select on; that is R4b), the medium, nuclear PDFs and isospin (a few %
  systematic, quoted separately).

## Settings
`config/OO_MCGlauber_MUSIC_isr_rhictune.xml`: the p+p tune's Pythia lines (`PDF:pSet = 20`,
$\alpha_s(M_Z) = 0.152$ for the hard process and the MPI, MPI `pTmin = 5`, ~1.7 GeV effective) with
Matter $Q_0 = 2$ GeV and `vir_factor` 0.25 — the tune's posterior at $Q_0 = 2$ (`conditional.py`).
p+p: the same XML with 3dMCGlauber's projectile and target set to p (this is also R3, the p+p reference
for O+O). $\hat p_T$ windows 5–80 GeV as in the tune; $\pi^0$ and $\eta$ undecayed; 900k p+p events and
300k O+O events in each impact-parameter class, $b$ = 0–2, 2–4, 4–6, 6–10 fm.

In [ ]:
import json, sys
import numpy as np
import matplotlib.pyplot as plt
import config as C, observables as O, tune as T

data = O.load_data()
S = json.loads((T.RES / "closure_oo.json").read_text())
smp = S["samples"]
U = np.load(T.RES / "closure_oo_units.npz")
CLASSES = [f"OO_b{lo}-{hi}" for lo, hi in S["b_classes"]]
COL = dict(zip(["pp"] + CLASSES, ["k", "C3", "C1", "C2", "C0"]))
LAB = {"pp": "p+p (same XML)", **{c: f"O+O, b {c[4:]} fm" for c in CLASSES}}
obs = {n: np.load(C.PROD_DIR / "obs" / f"closure_{n}.npz") for n in ["pp"] + CLASSES
       if (C.PROD_DIR / "obs" / f"closure_{n}.npz").exists()}
tune_runs = {n: np.load(C.obs_path(n)) for n in ("best", "ref") if C.obs_path(n).exists()}
print(f"{'sample':12s} {'events':>8s} {'<b>':>6s} {'<Ncoll>':>8s} {'<Npart>':>8s} {'<strings>':>9s}"
      f" {'skipped':>8s} {'x_hard msgs':>12s}")
for n in ["pp"] + CLASSES:
    s = smp[n]
    print(f"{n:12s} {int(sum(s['n_events'])):8d} {s['mean_glauber_b']:6.2f} {s['mean_glauber_ncoll']:8.2f}"
          f" {s['mean_glauber_npart']:8.2f} {s['mean_glauber_nstrings']:9.2f} {s['skipped']:8d} {s['xhard_msgs']:12d}")

## R3: the p+p reference in the O+O configuration, against the data
The same observables and $\chi^2$ as in the tune (`tune_comparison.ipynb`): PHENIX $\pi^0$ and STAR jets in
the fit ranges (35 bins, data stat ⊕ sys, normalisation uncertainties as fully correlated), STAR $\pi^0$
as validation. For comparison the tune's validation runs `ref` (0.150, 0.25, 2.0) and `best` (0.153,
0.319, 1.52), made with the p+p template XML. The p+p reference here differs from `ref` only by
$\alpha_s$ 0.152 instead of 0.150 and by coming from the O+O XML.

In [ ]:
ppall_path = C.PROD_DIR / "obs" / "closure_pp_all.npz"
runs3 = {"pp (R3, 900k)": obs["pp"]}
if ppall_path.exists():
    runs3["pp (R3, + 1.8M in 5-23 GeV)"] = np.load(ppall_path)
runs3.update(tune_runs)
blocks = []
for o in C.FIT:
    m = T.fit_mask(o, data); d = np.asarray(data[o]["value"])[m]
    rel2 = (np.asarray(data[o]["stat"])[m] ** 2 + np.asarray(data[o]["sys"])[m] ** 2) / d ** 2
    blocks.append((o, m, d, rel2, data[o]["norm"] or 0.0))
for name, ob in runs3.items():
    tot, nb, parts = 0.0, 0, []
    for o, m, d, rel2, norm in blocks:
        v, ve = O.combine(ob, o, data); r = np.log(v[m] / d)
        Cm = np.diag(rel2 + (ve[m] / v[m]) ** 2) + norm ** 2
        c = float(r @ np.linalg.solve(Cm, r)); tot += c; nb += m.sum(); parts.append(f"{o} {c:5.1f}/{m.sum()}")
    print(f"{name:9s} chi2 = {tot:5.1f} / {nb} bins   (" + ", ".join(parts) + ")")

fig, axs = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, (o, pmin) in zip(axs, (("phenix_pi0", 4), ("star_pi0", 4), ("star_jets_eta0", 6), ("star_jets_eta1", 6))):
    e = np.asarray(data[o]["edges"]); x = e.mean(1); d = np.asarray(data[o]["value"])
    err = np.hypot(data[o]["stat"], data[o]["sys"]) / d; sel = x >= pmin
    ax.fill_between(x[sel], 1 - err[sel], 1 + err[sel], color="0.85", step="mid", label="data stat⊕sys")
    for (name, ob), col, ls in zip(runs3.items(), ("C1", "C3", "C2", "C0"), ("-", "-", "--", ":")):
        v, ve = O.combine(ob, o, data); m = sel & (v > 0)
        ax.errorbar(x[m], (v / d)[m], (ve / d)[m], fmt=ls, color=col, label=name, lw=1.3)
    if o in C.FIT:
        ax.axvspan(*C.FIT[o], color="C0", alpha=0.05, label="fit range")
    ax.axhline(1, color="k", lw=0.6); ax.set_ylim(0.3, 1.9); ax.set_xlabel(r"$p_T$ [GeV]")
    ax.set_title(data[o]["label"], fontsize=9)
axs[0].set_ylabel("model / data"); axs[0].legend(fontsize=7); fig.tight_layout()

### R3: the $\pi^0$ with more statistics, and the run-to-run scatter of its $\chi^2$
1.8M more p+p events in the windows 5–23 GeV (`pp_low`, 300k per window instead of 90k), analysed together
with the 900k-event run. To see whether the model's $\pi^0$ errors are right, the combined sample is also
split into 4 independent quarters (jobs round-robin): if the model's statistical errors are correct, the
quarters scatter around the combined result as their errors say, and their $\pi^0$ $\chi^2$ (which includes
the model error) stays near the number of bins.

In [ ]:
splits = [np.load(C.PROD_DIR / "obs" / f"closure_pp_split{i}.npz") for i in range(4)
          if (C.PROD_DIR / "obs" / f"closure_pp_split{i}.npz").exists()]
def chi2_parts(ob):
    out = {}
    for o, m, d, rel2, norm in blocks:
        v, ve = O.combine(ob, o, data); r = np.log(v[m] / d)
        Cm = np.diag(rel2 + (ve[m] / v[m]) ** 2) + norm ** 2
        out[o] = float(r @ np.linalg.solve(Cm, r))
    return out
rows = {"combined": chi2_parts(runs3.get("pp (R3, + 1.8M in 5-23 GeV)", obs["pp"]))}
for i, s in enumerate(splits):
    rows[f"quarter {i}"] = chi2_parts(s)
print(f"{'':10s}" + "".join(f"{o:>16s}" for o in C.FIT) + f"{'total':>9s}")
for k, r in rows.items():
    print(f"{k:10s}" + "".join(f"{r[o]:16.1f}" for o in C.FIT) + f"{sum(r.values()):9.1f}")
o = "phenix_pi0"; e = np.asarray(data[o]["edges"]); x = e.mean(1); d = np.asarray(data[o]["value"])
fig, axs = plt.subplots(1, 2, figsize=(13, 3.8))
for i, s in enumerate(splits):
    v, ve = O.combine(s, o, data); m = v > 0
    axs[0].errorbar(x[m] * (1 + 0.006 * (i - 1.5)), (v / d)[m], (ve / d)[m], fmt="o", ms=3, color=f"C{i}", label=f"quarter {i}")
vc, vce = O.combine(runs3.get("pp (R3, + 1.8M in 5-23 GeV)", obs["pp"]), o, data)
axs[0].errorbar(x, vc / d, vce / d, fmt="k-", lw=1.5, label="combined")
err = np.hypot(data[o]["stat"], data[o]["sys"]) / d
axs[0].fill_between(x, 1 - err, 1 + err, color="0.85", step="mid", label="data stat⊕sys")
axs[0].axvspan(*C.FIT[o], color="C0", alpha=0.05); axs[0].axhline(1, color="k", lw=0.6)
axs[0].set_ylim(0.4, 2.0); axs[0].set_xlabel(r"$p_T$ [GeV]"); axs[0].set_ylabel("model / data"); axs[0].legend(fontsize=7)
axs[0].set_title("PHENIX $\\pi^0$: quarters of the R3 sample", fontsize=9)
# pulls of the quarters against the combined, in the fit range: (q - c) / sqrt(eq^2 - ec^2)
m = T.fit_mask(o, data); pulls = []
for s in splits:
    v, ve = O.combine(s, o, data)
    pulls.append(((v - vc) / np.sqrt(np.maximum(ve ** 2 - vce ** 2, 1e-300)))[m])
pulls = np.concatenate(pulls)
axs[1].hist(pulls, np.linspace(-4, 4, 17), histtype="step", color="k", density=True, label=f"quarters vs combined, rms {pulls.std():.2f}")
xx = np.linspace(-4, 4, 200); axs[1].plot(xx, np.exp(-xx ** 2 / 2) / np.sqrt(2 * np.pi), "C3--", label="unit Gaussian")
axs[1].set_xlabel("pull"); axs[1].legend(fontsize=7); axs[1].set_title("are the model's $\\pi^0$ errors right?", fontsize=9)
fig.tight_layout()
print(f"pi0 pulls of the quarters against the combined sample (fit range): mean {pulls.mean():+.2f}, rms {pulls.std():.2f} ({len(pulls)} values)")


## R4, event level: what O+O does to the hard nucleon pair
Per impact-parameter class (a random 60k-event subsample of each): the impact parameter, $N_\mathrm{coll}$ and
3dMCGlauber's strings (made in vacuum too, though no hydro uses them), and the energy bookkeeping of the
hard pair — the energy the hard scattering takes from the two nucleons (`isr_sub_E_total`: the initiators'
energy after ISR) and what is left of them as remnants for iColored (`remnant_E_proj + remnant_E_targ`). The
subtracted energy is fixed by the hard process and should not depend on the class; the remnants are where
the other collisions of the hard nucleons show up.

In [ ]:
def u(n, k): return U[f"{n}__{k}"].astype(float)
fig, axs = plt.subplots(2, 3, figsize=(16, 7.5))
for n in ["pp"] + CLASSES:
    c = COL[n]; l = LAB[n]
    if n != "pp":
        axs[0, 0].hist(u(n, "glauber_b"), np.linspace(0, 10, 51), histtype="step", color=c, density=True, label=l)
    axs[0, 1].hist(u(n, "glauber_ncoll"), np.arange(0.5, 40.5), histtype="step", color=c, density=True, label=l)
    axs[0, 2].hist(u(n, "glauber_nstrings"), np.arange(-0.5, 60.5), histtype="step", color=c, density=True, label=l)
    rem = u(n, "remnant_E_proj") + u(n, "remnant_E_targ")
    axs[1, 0].hist(rem, np.linspace(0, 200, 81), histtype="step", color=c, density=True, label=l)
    pth = u(n, "pthat"); sub = u(n, "isr_sub_E_total")
    be = np.array([5, 7, 9, 11, 14, 18, 23, 30, 40, 55, 80.])
    idx = np.digitize(pth, be) - 1
    xc = 0.5 * (be[:-1] + be[1:])
    for ax, val in ((axs[1, 1], sub), (axs[1, 2], rem)):
        mean = [np.nanmean(val[idx == i]) if (idx == i).sum() > 20 else np.nan for i in range(len(xc))]
        ax.plot(xc, mean, "o-", color=c, ms=3, label=l)
axs[0, 0].set_xlabel("b [fm]"); axs[0, 1].set_xlabel(r"$N_\mathrm{coll}$"); axs[0, 1].set_yscale("log")
axs[0, 2].set_xlabel("3dMCGlauber strings"); axs[0, 2].set_yscale("log")
axs[1, 0].set_xlabel("hard pair: remnant energy (proj + targ) [GeV]")
axs[1, 1].set_xlabel(r"$\hat p_T$ [GeV]"); axs[1, 1].set_ylabel("subtracted energy (initiators) [GeV]")
axs[1, 2].set_xlabel(r"$\hat p_T$ [GeV]"); axs[1, 2].set_ylabel("hard pair: remnant energy [GeV]")
for ax in axs.flat: ax.legend(fontsize=7)
for ax in (axs[1, 1], axs[1, 2]): ax.set_xscale("log")
fig.tight_layout()

## R4, the closure: O+O / p+p per hard scattering
Top: the spectra (cross section per hard scattering, the same windows and normalisation as the tune).
Bottom: O+O / p+p per class; 1 means the O+O environment leaves the hard event's hadrons and jets
unchanged. $\pi^0$ in PHENIX's binning ($|y|<0.35$), charged hadrons ($|\eta|<1$), STAR's jets
($R = 0.5$, off-axis-cone UE subtraction) at $|\eta|<0.5$ and $0.5<|\eta|<0.9$. Statistical errors only.

In [ ]:
def charged(n):
    s = smp[n]; return np.asarray(s["ch"]), np.asarray(s["ch_err"])
che = np.asarray(S["ch_edges"])
panels = [("phenix_pi0", r"$\pi^0$, $|y|<0.35$ (PHENIX binning)", 4),
          ("charged", r"charged hadrons, $|\eta|<1$", 2),
          ("star_jets_eta0", r"jets $R=0.5$, $|\eta|<0.5$", 6),
          ("star_jets_eta1", r"jets $R=0.5$, $0.5<|\eta|<0.9$", 6)]
def spec(n, o):
    if o == "charged":
        v, e = charged(n); x = 0.5 * (che[:-1] + che[1:]); return x, v, e
    ee = np.asarray(data[o]["edges"]); v, e = O.combine(obs[n], o, data); return ee.mean(1), v, e
fig, axs = plt.subplots(2, 4, figsize=(19, 7), gridspec_kw={"height_ratios": [1.6, 1]}, sharex="col")
summary_ratio = {}
for k, (o, title, pmin) in enumerate(panels):
    x, vp, ep = spec("pp", o); sel = (x >= pmin) & (vp > 0)
    axs[0, k].errorbar(x[sel], vp[sel], ep[sel], fmt="k-", lw=1.4, label=LAB["pp"])
    for n in CLASSES:
        _, v, e = spec(n, o); m = sel & (v > 0)
        axs[0, k].errorbar(x[m], v[m], e[m], fmt="o", ms=3, color=COL[n], label=LAB[n])
        r = v / vp; re = r * np.hypot(e / np.where(v > 0, v, 1), ep / vp)
        axs[1, k].errorbar(x[m], r[m], re[m], fmt="o-", ms=3, color=COL[n], lw=0.8)
        w = 1 / re[m] ** 2
        summary_ratio[(o, n)] = (np.sum(w * r[m]) / np.sum(w), 1 / np.sqrt(np.sum(w)))
    axs[0, k].set_yscale("log"); axs[0, k].set_title(title, fontsize=10)
    axs[1, k].axhline(1, color="k", lw=0.6); axs[1, k].set_ylim(0.8, 1.2)
    axs[1, k].set_xlabel(r"$p_T$ [GeV]"); axs[1, k].set_xscale("log")
axs[1, 0].set_ylabel("O+O / p+p (per hard scattering)"); axs[0, 0].legend(fontsize=7)
axs[0, 0].set_ylabel(r"$E\,d^3\sigma/dp^3$ [mb GeV$^{-2}$]"); axs[0, 1].set_ylabel(r"$d^2\sigma/dp_Td\eta$ [mb/GeV]")
axs[0, 2].set_ylabel(r"$d^2\sigma/dp_Td\eta$ [pb/GeV]")
fig.tight_layout()
print(f"{'O+O / p+p, weighted mean over the plotted range':48s}" + "".join(f"{c:>16s}" for c in CLASSES))
for o, title, _ in panels:
    print(f"{o:48s}" + "".join(f"{summary_ratio[(o, c)][0]:9.3f}±{summary_ratio[(o, c)][1]:.3f}" for c in CLASSES))

### Cross-check at the parton level
The hard process itself should be identical in O+O and p+p (the same Pythia, the same windows): the
cross-section-weighted spectrum of the leading outgoing parton of the hard scattering (`parton_pt_lead`, before
the final-state shower) and its rapidity, from the 60k-event subsamples. A deviation here would point to the
event bookkeeping (windows, weights, skipped events), not to the remnants or the hadronization.

In [ ]:
be = np.array([5, 6, 7, 8, 9, 10, 11, 12.5, 14, 16, 18, 20.5, 23, 26.5, 30, 35, 40, 47.5, 55, 67, 80])
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
sig = {n: np.asarray(smp[n]["sigma"]) for n in ["pp"] + CLASSES}
def wspec(n):
    pt, wi = u(n, "parton_pt_lead"), u(n, "pthat_bin").astype(int)
    nk = np.bincount(wi, minlength=len(sig[n])); w = sig[n][wi] / nk[wi]
    h = np.histogram(pt, be, weights=w)[0]; h2 = np.histogram(pt, be, weights=w ** 2)[0]
    return h / np.diff(be), np.sqrt(h2) / np.diff(be)
hp, ep = wspec("pp"); xc = 0.5 * (be[:-1] + be[1:])
for n in CLASSES:
    h, e = wspec(n); r = h / hp
    axs[0].errorbar(xc, r, r * np.hypot(e / h, ep / hp), fmt="o-", ms=3, color=COL[n], label=LAB[n])
    axs[1].hist(u(n, "parton_y_lead"), np.linspace(-3, 3, 41), histtype="step", density=True, color=COL[n], label=LAB[n])
axs[1].hist(u("pp", "parton_y_lead"), np.linspace(-3, 3, 41), histtype="step", density=True, color="k", lw=1.5, label=LAB["pp"])
axs[0].axhline(1, color="k", lw=0.6); axs[0].set_ylim(0.8, 1.2); axs[0].set_xscale("log")
axs[0].set_xlabel(r"leading hard parton $p_T$ [GeV]"); axs[0].set_ylabel("O+O / p+p (σ-weighted)")
axs[1].set_xlabel("leading hard parton $y$"); axs[1].set_ylabel("normalised")
for ax in axs: ax.legend(fontsize=7)
fig.tight_layout()

## Where the hard event's particles go: $dN_{ch}/d\eta$ and the remnants
Full-$\eta$ samples (no $\eta$ cut, iColored's input stored; 20k events at $\hat p_T$ 9–11 and 23–30 GeV per
class): the charged particles of the hard event (shower + remnants through iColored; there is no bulk in
vacuum), and the rapidity of the partons iColored fragments. In O+O the hard nucleons have had other
collisions, so their remnants carry less energy; this is where a difference from p+p is expected, and it
matters for R4b, since forward multiplicity is what STAR's centrality estimators count.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4))
for n in ["pp"] + CLASSES:
    s = smp.get(n + "_eta", {}).get("eta")
    if not s: continue
    ed = np.asarray(s["edges"]); xc = 0.5 * (ed[:-1] + ed[1:])
    lw = 1.6 if n == "pp" else 1.1
    axs[0].plot(xc, s["9-11"]["dndeta"], color=COL[n], lw=lw, label=LAB[n])
    axs[1].plot(xc, s["23-30"]["dndeta"], color=COL[n], lw=lw, label=LAB[n])
    axs[2].plot(xc, s["9-11"]["parton_dndy"], color=COL[n], lw=lw, label=LAB[n])
for ax, t in zip(axs, (r"$\hat p_T$ 9–11 GeV", r"$\hat p_T$ 23–30 GeV", r"$\hat p_T$ 9–11 GeV")):
    ax.set_xlim(-7, 7); ax.set_ylim(0); ax.set_title(t, fontsize=9); ax.legend(fontsize=7)
    ax.axvspan(2.1, 5.1, color="C0", alpha=0.06); ax.axvspan(-5.1, -2.1, color="C0", alpha=0.06)
axs[0].set_ylabel(r"hard event: $dN_{ch}/d\eta$ per event"); axs[0].set_xlabel(r"$\eta$"); axs[1].set_xlabel(r"$\eta$")
axs[2].set_xlabel("parton rapidity $y$"); axs[2].set_ylabel("iColored input: partons $dN/dy$")
fig.tight_layout()
print("shaded: 2.1 < |eta| < 5.1, the acceptance of STAR's EPD (forward centrality estimator)")
for n in ["pp"] + CLASSES:
    s = smp.get(n + "_eta", {}).get("eta")
    if not s: continue
    ed = np.asarray(s["edges"]); xc = 0.5 * (ed[:-1] + ed[1:]); w = np.diff(ed)
    for win in ("9-11", "23-30"):
        y = np.asarray(s[win]["dndeta"])
        print(f"{n:12s} {win:6s} dN/deta |eta|<1: {y[np.abs(xc) < 1].mean():5.2f}   "
              f"N in 2.1<|eta|<5.1: {np.sum((y * w)[(np.abs(xc) > 2.1) & (np.abs(xc) < 5.1)]):5.2f}   "
              f"total: {np.sum(y * w):5.1f}")

## Results and what they mean
Model results (this notebook's runs: 900k p+p and 4 × 300k O+O events in vacuum, $\hat p_T$ 5–80 GeV;
statistical errors only), not vetted physics results.

**R3, the p+p reference in the O+O configuration.** The jets agree with the tune's runs and the data
(jets $\chi^2$ 7.7 + 4.8 / 22 bins, against 9.8 + 4.4 for `ref`; 2–3 % above `ref`, as $\alpha_s$ 0.152 instead
of 0.150 implies). For the PHENIX $\pi^0$, 1.8M more events in the windows 5–23 GeV halve the model's error
(median 8.9 % → 4.6 % per bin), but the $\pi^0$ $\chi^2$ only drops from 29.5 to 25.3 / 13 (total 37.8 / 35):
the model is above PHENIX by 1.2–1.6 from 7 to 17 GeV, the known shape tension (shared by PP19, Pythia
default and Detroit, which are all ~0.7 of STAR's $\pi^0$ there), not statistics. The four independent
quarters of the sample scatter as their errors say ($\pi^0$ pulls against the combined sample: rms 1.12, mean
−0.24; quarter $\chi^2$ 17.6–28.5), so the model's errors are about right (perhaps ~10 % small), and
`ref`'s 10.3 was a low fluctuation of a run with 8–9 % $\pi^0$ errors. The job XMLs of R3 and `ref` differ
only in Matter's medium parameters (inactive in vacuum) and in values the driver overwrites per job, so
they run the same chain. As the p+p reference for O+O $R_{AA}$ this run is adequate: the same chain as O+O,
the jets as in the tune, and the $\pi^0$ normalisation cancels in $R_{AA}$; the $\pi^0$ shape tension is a
PHENIX-vs-STAR question, not one of the reference.

**R4, the vacuum closure.**
- **The hard process is the same in O+O and p+p** (leading-parton spectrum and rapidity: ratio 1 within
  the statistics), and so is the energy it takes from the two nucleons (the subtracted energy vs $\hat p_T$
  is identical in all classes), as built.
- **The hard yields close.** Per hard scattering, O+O / p+p is 1.00–1.02 for jets in both $\eta$ ranges in
  every class; the charged hadrons at $|\eta|<1$ are 1.00 (peripheral) to 1.03 (central, up to +5–8 % at
  3–6 GeV); the $\pi^0$ is 0.90–1.00 with ±4 % errors (consistent with 1 at 1–3 σ). The small central
  excess of charged hadrons is a hadronization effect of the weaker remnants (below), not of the hard
  process.
- **Failures are negligible:** 13 skipped events out of 300k in central O+O (16 in all classes), and no
  "Can not find x > x_hard" message with the tune's MPI (the earlier messages came with full soft MPI).
- **Where O+O differs: the hard nucleons' remnants.** In central O+O the hard pair's remnant energy is
  ~65 GeV instead of ~115 GeV in p+p at $\hat p_T$ 10 GeV (their other collisions' strings take the rest); the
  remnant partons sit at lower rapidity, and the hard event puts ~20 % fewer charged particles into
  $2.1<|\eta|<5.1$ (STAR's EPD acceptance: 9.6 instead of 12.0 per event at $\hat p_T$ 9–11 GeV), with the
  mid-rapidity yield unchanged.

**What it means.** In this chain energy conservation does not touch the hard yield at fixed impact parameter
(by construction, and the closure confirms that nothing else does, within ~2 % for jets and ~3 % for charged
hadrons); N$_\mathrm{coll}$ scaling holds per hard scattering, and the O+O $R_{AA}$ at fixed geometry measures the
medium. Energy conservation acts on the soft side instead: the hard nucleons carry less energy forward. That
is the ingredient of the centrality bias (R4b): with hydro, the subtraction also weakens the hard nucleons'
strings, and jet events then put less multiplicity into a forward estimator — the effect R4b has to
quantify with multiplicity classes. This vacuum test shows the hard event's own share of it: −20 % in the
EPD acceptance in central O+O, a small part of the total forward multiplicity there.


## R4b, part 1: a first look at the centrality bias (proxy, no hydro)
`centrality_proxy.py`. Experiments select centrality by the multiplicity in an estimator acceptance and
normalise with the minimum-bias $\langle N_\mathrm{coll}\rangle$ of that class. Here, before any hydro, a
proxy for that multiplicity: 3dMCGlauber's strings as MUSIC deposits them (constant local energy per
unit $\eta_s$ between the string ends, plus the remnant ends' rest energy), summed in STAR's windows —
$|\eta|<1$ (mid-rapidity multiplicity) and $2.1<|\eta|<5.1$ (EPD).

- **Minimum bias** (200k events): 3dMCGlauber alone, no hard scattering, strings made normally;
  its proxy distribution defines the percentile classes, as in the experiments.
- **Jet events** (2 × 200k): the ISR chain in vacuum, $b$ 0–10 fm, $\hat p_T$ 9–11, 18–23, 30–40,
  40–55 GeV, each event weighted with its $N_\mathrm{coll}$ (hard-scattering probability), with the
  hard-momentum subtraction on (`jet_sub`) and off (`jet_nosub`).
- **Experiment-style $R_{AA}^\mathrm{vac}$ per class** $= (f_{\mathrm{jet},c}/f_{\mathrm{MB},c})\,
  \langle N_\mathrm{coll}\rangle_\mathrm{MB}/\langle N_\mathrm{coll}\rangle_{\mathrm{MB},c}$: since the
  hard yield per scattering is unchanged (R4 above), any deviation from 1 is selection bias.

What the proxy leaves out: the hydro response and particlization fluctuations (they smear the
classes), and the jet's own hadrons in the estimator (at mid-rapidity the autocorrelation, which
pushes jet events to more central classes; small in the EPD, see R4). And it holds for Chun Shen's
string cut (`09e0a7e`) as it is now: the hard pair makes no string of its own.

In [ ]:
import importlib, glob
import centrality_proxy as P
cp = json.loads((T.RES / "centrality_proxy.json").read_text())
mb = {}
for f in sorted((P.OUT / "mb").glob("mb_seed*.npz")):
    d = np.load(f)
    for k in d.files: mb.setdefault(k, []).append(d[k])
mb = {k: np.concatenate(v) for k, v in mb.items()}
J = {n: P.load_jet(n) for n in ("jet_sub", "jet_nosub")}
nb_edges = np.array([1, 2, 3, 5, 9, 16, 26, 41])
xc = 0.5 * (nb_edges[:-1] + nb_edges[1:] - 1)
fig, axs = plt.subplots(2, 3, figsize=(17, 8))
for col, (e, lab) in enumerate((("epd", r"EPD $2.1<|\eta|<5.1$"), ("mid10", r"$|\eta|<1$"))):
    ax = axs[0, col]
    for d, kn, ke, c, l in ((mb, "ncoll", "eloc_", "k", "minimum bias (no hard scattering)"),
                            (J["jet_nosub"], "glauber_ncoll", "strings_eloc_", "C0", "jet events, no subtraction"),
                            (J["jet_sub"], "glauber_ncoll", "strings_eloc_", "C3", "jet events, subtraction")):
        idx = np.digitize(d[kn], nb_edges) - 1
        ax.plot(xc, [d[ke + e][idx == i].mean() for i in range(len(xc))], "o-", color=c, label=l)
    ax.set_xscale("log"); ax.set_xlabel(r"$N_\mathrm{coll}$"); ax.set_ylabel(f"string local energy in {lab} [GeV]")
    ax.legend(fontsize=7); ax.set_title("proxy vs $N_\\mathrm{coll}$", fontsize=9)
    ax = axs[1, col]
    labs = [f"{lo}-{hi}%" for lo, hi in cp["classes"]]
    for name, ls in (("jet_sub", "-"), ("jet_nosub", "--")):
        for (wk, rw), c in zip(cp["jet"][name]["per_window"].items(), ("C0", "C1", "C2", "C3")):
            ax.plot(range(len(labs)), rw[e]["raa_vac"], ls, marker="o", ms=3, color=c,
                    label=f"{'sub' if name == 'jet_sub' else 'no sub'}, $\\hat p_T$ {wk}")
    ax.axhline(1, color="k", lw=0.6); ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs)
    ax.set_ylabel(r"experiment-style $R_{AA}^\mathrm{vac}$"); ax.set_title(f"classes by {lab}", fontsize=9)
    ax.set_ylim(0.6, 2.5); ax.legend(fontsize=6, ncol=2)
ax = axs[0, 2]
for d, kn, ks, c, l in ((mb, "ncoll", "nstrings", "k", "minimum bias"), (J["jet_nosub"], "glauber_ncoll", "glauber_nstrings", "C0", "jets, no sub"),
                        (J["jet_sub"], "glauber_ncoll", "glauber_nstrings", "C3", "jets, sub")):
    idx = np.digitize(d[kn], nb_edges) - 1
    ax.plot(xc, [d[ks][idx == i].mean() - xc[i] for i in range(len(xc))], "o-", color=c, label=l)
ax.set_xscale("log"); ax.set_xlabel(r"$N_\mathrm{coll}$"); ax.set_ylabel(r"strings $-$ bin centre $N_\mathrm{coll}$")
ax.legend(fontsize=7); ax.set_title("strings per event (offset by $N_\\mathrm{coll}$)", fontsize=9)
ax = axs[1, 2]
x = mb["eloc_epd"]; ax.hist(x, np.linspace(0, 250, 101), density=True, histtype="step", color="k", label="minimum bias")
w = J["jet_sub"]["glauber_ncoll"].astype(float)
ax.hist(J["jet_sub"]["strings_eloc_epd"], np.linspace(0, 250, 101), weights=w, density=True, histtype="step", color="C3", label="jets, sub (N_coll-weighted)")
ax.hist(J["jet_nosub"]["strings_eloc_epd"], np.linspace(0, 250, 101), weights=J["jet_nosub"]["glauber_ncoll"].astype(float),
        density=True, histtype="step", color="C0", label="jets, no sub (N_coll-weighted)")
for cut in cp["mb"]["epd"]["cuts"][:-1]: ax.axvline(cut, color="0.7", lw=0.6)
ax.set_yscale("log"); ax.set_xlabel("EPD proxy [GeV]"); ax.legend(fontsize=7); ax.set_title("class boundaries (grey) from minimum bias", fontsize=9)
fig.tight_layout()
print(f"{'':28s}" + "".join(f"{l:>9s}" for l in labs))
for name in ("jet_sub", "jet_nosub"):
    for e in ("epd", "mid10"):
        v = np.mean([rw[e]["raa_vac"] for rw in cp["jet"][name]["per_window"].values()], axis=0)
        print(f"{name:10s} {e:6s} (mean of windows)" + "".join(f"{x:9.2f}" for x in v))


**Results (proxy).** With the current set-up, experiment-style classes would show a strong selection
bias in O+O even without any medium:
- **Classes by the EPD:** $R_{AA}^\mathrm{vac}$ ≈ 0.80 in 0–10 %, 1.0–1.15 from 10 to 80 %, ≈ 2.3 in 80–100 %,
  nearly independent of $\hat p_T$ (9–55 GeV). Without the subtraction: 0.95 and 1.4.
- **Classes by $|\eta|<1$:** ≈ 0.87 in 0–10 % and ≈ 1.8 in 80–100 %, with or without the subtraction.
- **Two causes, visible at fixed $N_\mathrm{coll}$ (top row):** (1) jet events have 1.1–1.3 fewer strings than
  minimum-bias events at the same $N_\mathrm{coll}$ (0.41 against 1.49 at $N_\mathrm{coll}=1$): the hard pair
  makes no string of its own (3dMCGlauber `09e0a7e`), −3 to −4 GeV in both windows, most important in
  peripheral events; (2) the subtraction takes a further ~6.5 GeV out of the EPD window, but only ~0.5 GeV
  at mid-rapidity, at every $N_\mathrm{coll}$. So the mid-rapidity bias is the string cut's, the EPD bias
  mostly the subtraction's.
- **Caveats:** a proxy (string energy, no hydro response or particlization fluctuations, which smear the
  classes and typically weaken the bias in the central class); the jet's own particles are missing (at
  mid-rapidity they push the other way); the most peripheral class is sensitive to events with few
  strings. The full test (O+O with hydro, the jet in vacuum, multiplicity classes) and the string switch
  will tell how much of this survives; the size here says it is worth doing.

### R4b, part 1b: sPHENIX's MBD window, its classes, and the minimum-bias coincidence
sPHENIX defines centrality from the MBD charge sum (3.51 < |η| < 4.61, both sides), measures the classes
0–20, 20–40, 40–60, 60–80 %, and requires at least one MBD hit on each side (minimum-bias trigger efficiency
91.4 % of σ_OO); its inclusive "0–100 %" yield is corrected for the efficiency of that coincidence with
Pythia + HIJING (sPH-CONF-JET-2026-02). The proxy, now also in the MBD window and per arm (`v2` samples, the
jet events with their charged hadrons to |η| < 5): R_AA^vac and R_CP in sPHENIX's classes, and the
coincidence efficiency — an arm fires if the strings deposit local energy above a threshold in it (and, for
jet events, also if a charged hadron of the hard event is in it).

In [ ]:
SPH = [(0, 20), (20, 40), (40, 60), (60, 80), (80, 100)]
def sph_classes(est):
    x = mb[f"eloc_{est}"]; cuts = np.percentile(x, [100 - hi for _, hi in SPH])
    def cls(v):
        c = np.full(len(v), len(SPH) - 1)
        for k in range(len(SPH) - 1, -1, -1): c[v >= cuts[k]] = k
        return c
    cm = cls(x); fmb = np.array([np.mean(cm == k) for k in range(len(SPH))])
    nmb = np.array([mb["ncoll"][cm == k].mean() for k in range(len(SPH))])
    res = {}
    for name in ("jet_sub", "jet_nosub"):
        d = J[name]; w = d["glauber_ncoll"].astype(float); cj = cls(d[f"strings_eloc_{est}"])
        fj = np.array([w[cj == k].sum() for k in range(len(SPH))]) / w.sum()
        res[name] = fj / fmb * mb["ncoll"].mean() / nmb
    return nmb, res
fig, axs = plt.subplots(1, 3, figsize=(17, 4))
labs = [f"{lo}-{hi}%" for lo, hi in SPH]
for ax, est, title in ((axs[0], "mbd", "classes by the MBD (sPHENIX)"), (axs[1], "epd", "classes by the EPD (STAR)")):
    nmb, res = sph_classes(est)
    for name, c, ls in (("jet_sub", "C3", "-"), ("jet_nosub", "C0", "--")):
        r = res[name]
        ax.plot(range(len(SPH)), r, ls, marker="o", color=c, label=("subtraction" if name == "jet_sub" else "no subtraction")
                + f": R_CP 0-20/60-80 {r[0] / r[3]:.2f}, 0-20/40-60 {r[0] / r[2]:.2f}")
    ax.axhline(1, color="k", lw=0.6); ax.set_xticks(range(len(SPH))); ax.set_xticklabels(labs); ax.set_ylim(0.6, 2.6)
    ax.set_ylabel(r"experiment-style $R_{AA}^\mathrm{vac}$ (proxy)"); ax.set_title(title, fontsize=9); ax.legend(fontsize=7)
    print(f"{est}: model minimum-bias <N_coll> per class " + " ".join(f"{v:.1f}" for v in nmb)
          + ("   (sPHENIX Table 1: 22.5 12.2 6.0 2.9)" if est == "mbd" else ""))
    for name in res:
        r = res[name]
        print(f"   {name:9s} R_AA^vac " + " ".join(f"{l} {v:.2f}" for l, v in zip(labs, r))
              + f" | R_CP 0-20/60-80 {r[0] / r[3]:.2f}, 20-40/60-80 {r[1] / r[3]:.2f}, 40-60/60-80 {r[2] / r[3]:.2f}, 0-20/40-60 {r[0] / r[2]:.2f}")
coin = cp["mbd_coincidence"]; nbins = coin["ncoll_bins"]; xl = [f"{lo}" if lo == hi else f"{lo}-{hi if hi < 1000 else ''}" for lo, hi in nbins]
ax = axs[2]
for thr, ls in zip(coin["thresholds_GeV"], ("-", "--", ":")):
    t = str(thr)
    ax.plot(range(len(nbins)), coin["mb"][t]["by_ncoll"], ls, marker="o", color="k", label=f"minimum bias, > {thr} GeV")
    ax.plot(range(len(nbins)), coin["jet"]["jet_sub"][t]["by_ncoll"], ls, marker="o", color="C3", label=f"jets, subtraction, > {thr} GeV")
    ax.plot(range(len(nbins)), coin["jet"]["jet_nosub"][t]["by_ncoll"], ls, marker="o", color="C0", label=f"jets, no subtraction, > {thr} GeV")
ax.set_xticks(range(len(nbins))); ax.set_xticklabels(xl); ax.set_xlabel(r"$N_\mathrm{coll}$"); ax.set_ylabel("MBD coincidence efficiency (proxy)")
ax.legend(fontsize=6); ax.set_ylim(0, 1.05); ax.set_title("≥ 1 hit in each MBD arm", fontsize=9)
fig.tight_layout()
for thr in coin["thresholds_GeV"]:
    t = str(thr); c = coin["jet"]
    print(f"arm threshold {thr} GeV: minimum bias {coin['mb'][t]['all']:.3f} (data: 0.914); jets N_coll-weighted: "
          f"subtraction {c['jet_sub'][t]['weighted']:.3f}, no subtraction {c['jet_nosub'][t]['weighted']:.3f}")


**Results (proxy, MBD).**
- The model's minimum-bias $\langle N_\mathrm{coll}\rangle$ in sPHENIX's MBD classes (22.8 / 12.4 / 6.1 / 2.8) agrees with
  sPHENIX's Glauber table (22.5 / 12.2 / 6.0 / 2.9).
- With the subtraction, classes by the MBD give $R_{AA}^\mathrm{vac}$ ≈ 0.86 / 1.04 / 1.09 / 1.15 (0–20 / 20–40 /
  40–60 / 60–80 %), i.e. $R_{CP}$(0–20/60–80) ≈ 0.74 and $R_{CP}$(0–20/40–60) ≈ 0.79 from the selection alone,
  flat in $\hat p_T$. sPHENIX estimates the bias with Angantyr at < 10 % on $R_{CP}$; it measures
  $R_{CP}$(0–20/60–80) ≈ 0.62–0.68 and $R_{AA}$ ≈ 0.7 / 0.9 / 1.05 / 1.05 (read from Figs. 5, 7). Taken at face
  value the proxy bias would be most of the measured $R_{CP}$, and per class the data would be the bias times a
  suppression of ≈ 0.8 (central) to ≈ 0.9–1 (peripheral); the inclusive $R_{AA}$ ≈ 0.8 is untouched by the bias.
- Without the subtraction the MBD classes show no bias (0.93–1.05): at 3.5 < |η| < 4.6 the string cut does not
  matter, the bias there is the subtraction's alone (unlike mid-rapidity, where it is the cut's).
- The MBD coincidence: in minimum bias the proxy gives 0.99 / 0.96 / 0.88 for arm thresholds of 0 / 0.5 / 1 GeV
  of string energy (sPHENIX: 0.914, i.e. a threshold near 0.8 GeV). Jet events (N_coll-weighted, with their
  own hadrons) pass at 0.98 / 0.975 / 0.967 with the subtraction, 0.998–1.0 without: the subtraction costs
  2–3 % of the jet yield through the coincidence. If the Pythia + HIJING correction misses it, the inclusive
  $R_{AA}$ is ≤ 3 % low — small against 0.8.
- **Caveat that matters most here:** the proxy has no multiplicity fluctuations. The MBD charge at a given
  energy deposit fluctuates (sPHENIX fits an NBD per ancestor), and in O+O's small multiplicities that smears
  the classes strongly and dilutes a selection bias, most in the central class. The 0.74 is therefore an upper
  bound on the effect (part 1c below: ~0.83 with fluctuations); the full test with hydro, particlization and the MBD charge (or an NBD smearing of the
  proxy as a quick bracket) decides how much remains.

### R4b, part 1c: fluctuations — negative-binomial smearing of the proxy (step 1 of the plan)
The detector counts at a given energy deposit fluctuate (sPHENIX fits its MBD charge with Glauber ⊕ NBD), and
in O+O's small multiplicities that smears the classes and dilutes a selection bias. Here each event's string
local energy per MBD arm (and in the EPD) becomes a count drawn from a negative binomial with mean $\mu E$ and
shape $\kappa E$ (independent NBDs per GeV); jet events add the charged hadrons of their hard event in the same
windows (particles, not smeared). $\mu$ and $\kappa$ are scanned; sPHENIX's minimum-bias coincidence efficiency
(≥ 1 count per arm, 0.914) picks the realistic combinations. Classes are percentiles of all minimum-bias events
(random tie-break), as anchored by a Glauber ⊕ NBD fit. `centrality_proxy.py smear`. Only the subtraction case
is meaningful: without the subtraction the hard pair's remnant would be counted twice (strings and iColored).

In [ ]:
sm = json.loads((T.RES / "centrality_proxy_smear.json").read_text())
rows = sm["grid"]
print(f"{'mu':>4s} {'kappa':>7s} {'eps_trig':>9s} | MBD classes (subtraction): R_AA^vac 0-20 20-40 40-60 60-80 | R_CP 0-20/60-80  0-20/40-60 | EPD R_CP")
for r in rows:
    m = r["mbd"]["jet_sub"]; e = r["epd"]["jet_sub"]
    flag = "  <- eps_trig within 0.85-0.94" if 0.85 <= r["eps_trig_mb"] <= 0.94 else ""
    print(f"{r['mu']:4.1f} {r['kappa']:7.1f} {r['eps_trig_mb']:9.3f} | " + " ".join(f"{v:5.2f}" for v in m["raa_vac"][:4])
          + f" | {m['rcp_0_20_60_80']:5.2f} {m['rcp_0_20_40_60']:5.2f} | {e['rcp_0_20_60_80']:5.2f}{flag}")
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for est, ax in (("mbd", axs[0]), ("epd", axs[1])):
    for mu, c in zip(sm["mu"], ("C7", "C0", "C2", "C3")):
        rr = [r for r in rows if r["mu"] == mu]
        ax.plot([r["eps_trig_mb"] for r in rr], [r[est]["jet_sub"]["rcp_0_20_60_80"] for r in rr], "o-", color=c,
                label=f"$\\mu$ = {mu}/GeV ($\\kappa$ 0.1 ... Poisson)")
    ax.axvline(0.914, color="k", ls="--", lw=0.8, label="sPHENIX $\\epsilon_{trig}$ 0.914")
    ax.axhline(1, color="k", lw=0.5); ax.set_xlabel("minimum-bias MBD coincidence efficiency (model)")
    ax.set_ylabel("selection-only $R_{CP}$(0-20/60-80)"); ax.set_ylim(0.6, 1.3); ax.legend(fontsize=7)
    ax.set_title(f"classes by the {est.upper()}, subtraction, NBD-smeared", fontsize=9)
fig.tight_layout()


**Results (step 1).** The combinations that reproduce sPHENIX's minimum-bias efficiency (0.85–0.94) are
$\mu$ ≈ 1–2 counts per GeV with $\kappa \gtrsim 1$ or Poisson. For them the selection-only bias in sPHENIX's MBD
classes is $R_{CP}$(0–20/60–80) ≈ 0.82–0.91 (closest to 0.914, $\mu$ = 2, $\kappa$ = 3: 0.83, with
$R_{AA}^\mathrm{vac}$ ≈ 0.91 / 1.03 / 1.05 / 1.10), and ≈ 0.86–0.92 in EPD classes — down from 0.74 / 0.78 without
fluctuations: the fluctuations and the jet's own forward hadrons (autocorrelation) dilute the bias to ~10–18 %.
That is somewhat above sPHENIX's Angantyr estimate (< 10 %) and well short of the measured $R_{CP}$ ≈ 0.62–0.68:
on this estimate the data are the bias (~0.85) times a suppression of ~0.75–0.8 in 0–20 % relative to 60–80 %.
Remaining caveats: a proxy (string energy, not hydro + particlization), $\mu$ anchored only through the
coincidence efficiency, the string cut's mid-rapidity part not involved in these forward windows. The full R4b
run measures it with real multiplicities.